# 🔍 Large-Scale Credit Card Fraud Detection via Spectral Clustering
## Power Iteration Clustering (PIC) on Apache Spark

---

### 📐 Mathematical Foundation

This notebook implements **Spectral Clustering** using the **Graph Laplacian** to detect fraud rings in a large-scale transaction graph — all powered by Apache Spark's distributed PIC algorithm.

#### Why Graph Theory for Fraud?

Credit card fraud rarely occurs in isolation. Fraudsters operate in **coordinated rings** — sharing cards, merchants, and compromised accounts. When modelled as a **weighted undirected graph** `G = (V, E, W)`, where:
- **`V` (vertices)** = account/merchant node IDs
- **`E` (edges)** = transactions between accounts
- **`W` (weights)** = normalised transaction amounts

...fraud rings appear as **densely connected, isolated subgraphs** — exactly what spectral methods are designed to find.

#### The Graph Laplacian

For a graph with adjacency matrix **W** and degree matrix **D** (where `D_ii = Σ_j W_ij`), the **unnormalised Graph Laplacian** is:

$$L = D - W$$

Its eigenvectors encode the **community structure** of the graph. Small eigenvalues of `L` correspond to eigenvectors that are nearly constant within clusters — the **Fiedler vector** (second-smallest eigenvector) partitions the graph optimally.

The **normalised Laplacian** used in practice:

$$L_{sym} = D^{-1/2} L D^{-1/2} = I - D^{-1/2} W D^{-1/2}$$

#### Power Iteration Clustering (PIC) — Lin & Cohen, 2010

Computing eigenvectors of `L` directly at petabyte scale is intractable. **PIC** approximates this via a normalised random walk on the graph. Starting from a uniform vector `v₀`, it iterates:

$$v^{(t+1)} = \frac{W \cdot v^{(t)}}{\| W \cdot v^{(t)} \|_1}$$

This converges to a **pseudo-eigenvector** that approximates the leading non-trivial eigenvector of the normalised adjacency matrix — equivalent to the Fiedler vector of `L_{sym}`. Nodes are then clustered by their scalar values in this vector using k-means.

**Key advantages over classical spectral methods:**
- `O(|E|)` per iteration instead of `O(|V|³)` for full eigendecomposition
- Fully parallelisable as sparse matrix-vector multiplications on Spark
- Scales to billions of edges

---

## 🏗️ Section 1: Infrastructure & SparkSession Configuration

### Architectural Rationale

PIC is memory-intensive and shuffle-heavy. Each iteration performs a **sparse matrix-vector multiply** (`W · v`) which requires shuffling edge data across the cluster. We configure Spark to handle this efficiently:

| Configuration | Value | Why |
|---|---|---|
| `spark.serializer` | `KryoSerializer` | 10x faster than Java serialization for graph edge objects |
| `spark.sql.shuffle.partitions` | `200` | Prevents OOM on large graph shuffles; tune to `2-3x cores` |
| `spark.executor.memory` | `4g` | Graph adjacency structures are memory-resident during iteration |
| `spark.driver.memory` | `2g` | Cluster summary stats pulled to driver for scoring |
| `spark.default.parallelism` | `200` | Ensures RDD operations match shuffle partition count |
| `spark.sql.adaptive.enabled` | `true` | AQE dynamically coalesces skewed partitions (dense fraud cliques can cause skew) |
| `spark.ml.pic.*` | *(via model params)* | Convergence tolerance and max iterations set on the estimator |

> **Note:** In a production cluster (e.g., Databricks / EMR), `executor.memory` and `parallelism` are controlled by cluster config. These local settings demonstrate the intent.

In [1]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

for var in ["SPARK_HOME", "HADOOP_CONF_DIR", "YARN_CONF_DIR", "PYTHONPATH", "PYSPARK_SUBMIT_ARGS", "SPARK_CONF_DIR"]:
    os.environ.pop(var, None)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType, IntegerType, StringType
)
from pyspark.ml.clustering import PowerIterationClustering

import pandas as pd
import numpy as np
import random
import itertools
import math

spark = (
    SparkSession.builder
    .appName("FraudDetection_SpectralClustering_PIC")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.unsafe", "true")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "4g")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.3")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "200")
    .config("spark.shuffle.compress", "true")
    .config("spark.shuffle.spill.compress", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.ml.powerIterationClustering.convergenceTol", "1e-5")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession initialised  |  Spark {spark.version}")
print(f"   App Name  : {spark.sparkContext.appName}")
print(f"   Master    : {spark.sparkContext.master}")
print(f"   Cores     : {spark.sparkContext.defaultParallelism}")

26/04/10 12:50:17 WARN Utils: Your hostname, LAPTOP-BBBI5KIQ resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/10 12:50:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 12:50:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession initialised  |  Spark 3.5.0
   App Name  : FraudDetection_SpectralClustering_PIC
   Master    : local[*]
   Cores     : 200


In [2]:
# ============================================================
# CELL 1.1 — Dependency imports & SparkSession initialisation
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType, IntegerType, StringType
)
from pyspark.ml.clustering import PowerIterationClustering

import pandas as pd
import numpy as np
import random
import itertools
import math

# ── SparkSession: tuned for graph/shuffle workloads ─────────────────────────
spark = (
    SparkSession.builder
    .appName("FraudDetection_SpectralClustering_PIC")

    # Kryo serialisation: critical for large graph edge objects
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.unsafe", "true")          # additional Kryo speed boost

    # Memory layout
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "4g")
    .config("spark.memory.fraction", "0.8")       # more heap for execution/storage
    .config("spark.memory.storageFraction", "0.3")

    # Shuffle tuning for large graph iterations
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "200")
    .config("spark.shuffle.compress", "true")
    .config("spark.shuffle.spill.compress", "true")

    # Adaptive Query Execution — handles partition skew from dense cliques
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")

    # Graph algorithm tolerance
    .config("spark.ml.powerIterationClustering.convergenceTol", "1e-5")

    .master("local[*]")   # Replace with yarn / k8s in production
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession initialised  |  Spark {spark.version}")
print(f"   App Name  : {spark.sparkContext.appName}")
print(f"   Master    : {spark.sparkContext.master}")
print(f"   Cores     : {spark.sparkContext.defaultParallelism}")

✅ SparkSession initialised  |  Spark 3.5.0
   App Name  : FraudDetection_SpectralClustering_PIC
   Master    : local[*]
   Cores     : 200


In [3]:
import os
import warnings
from pyspark.sql import SparkSession

warnings.filterwarnings("ignore")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = (
    SparkSession.builder
    .appName("FraudDetection_Test")
    .master("local[*]")
    .getOrCreate()
)

print(spark.version)

3.5.0


26/04/10 12:50:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---

## 🎲 Section 2: Synthetic Transaction Graph Generation

### Data Model & Graph Construction

We model transactions as a **weighted, undirected edge list** — the canonical input format for PIC:

```
src (LongType) | dst (LongType) | weight (DoubleType)
```

> **⚠️ PIC Constraint:** `pyspark.ml.clustering.PowerIterationClustering` requires `src` and `dst` to be `LongType`. Using `IntegerType` or `StringType` will raise a runtime error. All node IDs must be cast explicitly.

### Fraud Ring Design (Ground Truth)

We embed **two hidden fraud rings** using a **clique structure**:

- A **clique** `K_n` (complete graph on `n` nodes) has `n(n-1)/2` edges — the maximum possible density.
- Fraud rings are cliques because every fraudulent actor transacts with every other (card sharing, circular laundering, split transactions).
- The weights within the ring are **uniformly high** (large transactions) with **low variance** — a signature of scripted fraud bots.
- Rings are **isolated**: no edges connect them to the normal transaction graph, making them appear as disconnected subgraphs with extremely low cut values.

The **Laplacian cut cost** for separating a clique from the rest of the graph:

$$\text{cut}(A, \bar{A}) = 0 \quad (\text{for isolated clique } A)$$

This makes them the *easiest possible* clusters to detect spectrally — the algorithm will assign them near-identical pseudo-eigenvector values, grouping them tightly in 1D scalar space.

In [4]:
import sys
import os
import pyspark
import subprocess

print("--- ENVIRONMENT DIAGNOSTICS ---")
print(f"Python version : {sys.version.split(' ')[0]}")
print(f"PySpark version: {pyspark.__version__}")
print(f"JAVA_HOME env  : {os.environ.get('JAVA_HOME', 'NOT SET')}")

try:
    java_info = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode("utf-8")
    print(f"Java executable: {java_info.splitlines()[0]}")
except Exception as e:
    print(f"Java executable: ERROR - {e}")

--- ENVIRONMENT DIAGNOSTICS ---
Python version : 3.11.15
PySpark version: 3.5.0
JAVA_HOME env  : /usr/lib/jvm/java-17-openjdk-amd64
Java executable: openjdk version "17.0.18" 2026-01-20


In [5]:
# ============================================================
# CELL 2.1 — Synthetic Data Generator
# ============================================================

class FraudGraphGenerator:
    """
    Generates a synthetic transaction edge list with embedded fraud rings.

    Parameters
    ----------
    n_normal_nodes   : int   — Number of legitimate account/merchant nodes.
    n_normal_edges   : int   — Number of random legitimate transactions.
    fraud_rings      : list  — Each dict: {'n_nodes': int, 'weight_mean': float,
                                           'weight_std': float, 'id_offset': int}
    seed             : int   — Random seed for reproducibility.
    """

    def __init__(self, n_normal_nodes=500, n_normal_edges=3000,
                 fraud_rings=None, seed=42):
        self.n_normal_nodes = n_normal_nodes
        self.n_normal_edges = n_normal_edges
        self.seed = seed
        random.seed(seed)
        np.random.seed(seed)

        # Default: two distinct fraud rings
        self.fraud_rings = fraud_rings or [
            {"n_nodes": 8,  "weight_mean": 950.0, "weight_std": 15.0,
             "id_offset": 10_000},   # Ring A: large-value, tight amounts
            {"n_nodes": 6,  "weight_mean": 480.0, "weight_std": 8.0,
             "id_offset": 20_000},   # Ring B: medium-value, very tight amounts
        ]

    # ── Normal transaction graph (Erdős–Rényi random graph) ─────────────────
    def _generate_normal_edges(self):
        edges = []
        node_ids = list(range(1, self.n_normal_nodes + 1))
        for _ in range(self.n_normal_edges):
            src = int(random.choice(node_ids))
            dst = int(random.choice(node_ids))
            if src == dst:
                continue
            # Transaction amounts: log-normal (typical spend distribution)
            weight = float(np.random.lognormal(mean=4.5, sigma=1.2))
            weight = max(round(weight, 2), 1.0)
            edges.append((src, dst, weight, "normal"))
        return edges

    # ── Fraud ring: fully connected clique K_n ───────────────────────────────
    def _generate_fraud_ring(self, ring_cfg, ring_label):
        edges = []
        n     = ring_cfg["n_nodes"]
        off   = ring_cfg["id_offset"]
        mu    = ring_cfg["weight_mean"]
        sigma = ring_cfg["weight_std"]
        node_ids = list(range(off, off + n))

        # Clique K_n: every pair connected — maximum density
        for src, dst in itertools.combinations(node_ids, 2):
            weight = float(max(np.random.normal(mu, sigma), 1.0))
            weight = round(weight, 2)
            edges.append((int(src), int(dst), weight, ring_label))
        return edges

    # ── Public: build the full edge list ────────────────────────────────────
    def generate(self):
        all_edges = self._generate_normal_edges()

        for i, ring in enumerate(self.fraud_rings):
            label = f"fraud_ring_{chr(65 + i)}"   # fraud_ring_A, fraud_ring_B …
            all_edges.extend(self._generate_fraud_ring(ring, label))
            n = ring["n_nodes"]
            clique_edges = n * (n - 1) // 2
            print(f"   💀 {label}: {n} nodes | {clique_edges} edges "
                  f"| node IDs [{ring['id_offset']} – {ring['id_offset'] + n - 1}] "
                  f"| avg weight ≈ ${ring['weight_mean']:.0f}")

        random.shuffle(all_edges)
        return all_edges


# ── Instantiate generator & build raw edges ──────────────────────────────────
print("📊 Generating synthetic transaction graph...")
generator = FraudGraphGenerator(
    n_normal_nodes=500,
    n_normal_edges=3000,
    seed=42
)

raw_edges = generator.generate()
print(f"\n✅ Total edges generated: {len(raw_edges):,}")

📊 Generating synthetic transaction graph...
   💀 fraud_ring_A: 8 nodes | 28 edges | node IDs [10000 – 10007] | avg weight ≈ $950
   💀 fraud_ring_B: 6 nodes | 15 edges | node IDs [20000 – 20005] | avg weight ≈ $480

✅ Total edges generated: 3,039


In [6]:
# ============================================================
# CELL 2.2 — Create Spark DataFrame with explicit schema
# ============================================================

# PIC mandates LongType for src/dst — define the schema explicitly
edge_schema = StructType([
    StructField("src",        LongType(),   nullable=False),
    StructField("dst",        LongType(),   nullable=False),
    StructField("weight",     DoubleType(), nullable=False),
    StructField("true_label", StringType(), nullable=True),   # ground truth for evaluation
])

edge_df = spark.createDataFrame(raw_edges, schema=edge_schema)

# ── Deduplication: keep max-weight edge for duplicate (src, dst) pairs ───────
# Undirected graphs can have (A→B) and (B→A); PIC expects a single edge per pair.
edge_df = (
    edge_df
    .withColumn("src_norm", F.least(F.col("src"), F.col("dst")))
    .withColumn("dst_norm", F.greatest(F.col("src"), F.col("dst")))
    .groupBy("src_norm", "dst_norm", "true_label")
    .agg(F.max("weight").alias("weight"))
    .withColumnRenamed("src_norm", "src")
    .withColumnRenamed("dst_norm", "dst")
    .filter(F.col("src") != F.col("dst"))   # remove self-loops
    .select("src", "dst", "weight", "true_label")
)

# ── Normalise weights to [0, 1] (PIC convention for stable convergence) ──────
w_max = edge_df.agg(F.max("weight")).collect()[0][0]
edge_df = edge_df.withColumn("weight_norm", F.round(F.col("weight") / w_max, 6))

# ── Cache: this DataFrame is read every PIC iteration ────────────────────────
edge_df.cache()
total_edges = edge_df.count()   # triggers cache materialisation

print(f"✅ Edge DataFrame cached")
print(f"   Total deduplicated edges : {total_edges:,}")
print(f"   Max raw weight (normaliser): ${w_max:,.2f}")
print(f"\n📐 Schema:")
edge_df.printSchema()

print("\n🔍 Sample edges (5 rows):")
edge_df.show(5, truncate=False)

✅ Edge DataFrame cached
   Total deduplicated edges : 3,000
   Max raw weight (normaliser): $10,011.46

📐 Schema:
root
 |-- src: long (nullable = false)
 |-- dst: long (nullable = false)
 |-- weight: double (nullable = true)
 |-- true_label: string (nullable = true)
 |-- weight_norm: double (nullable = true)


🔍 Sample edges (5 rows):
+---+---+------+----------+-----------+
|src|dst|weight|true_label|weight_norm|
+---+---+------+----------+-----------+
|236|290|48.92 |normal    |0.004886   |
|62 |483|203.47|normal    |0.020324   |
|57 |217|378.38|normal    |0.037795   |
|73 |296|53.3  |normal    |0.005324   |
|95 |323|776.08|normal    |0.077519   |
+---+---+------+----------+-----------+
only showing top 5 rows



In [7]:
# ============================================================
# CELL 2.3 — Exploratory graph statistics
# ============================================================

print("═" * 60)
print("  GRAPH SUMMARY STATISTICS")
print("═" * 60)

# Unique nodes (union of src and dst sets)
src_nodes = edge_df.select(F.col("src").alias("node_id"))
dst_nodes = edge_df.select(F.col("dst").alias("node_id"))
all_nodes = src_nodes.union(dst_nodes).distinct()
n_nodes   = all_nodes.count()

# Edge distribution by label
label_counts = (
    edge_df
    .groupBy("true_label")
    .agg(
        F.count("*").alias("edge_count"),
        F.round(F.avg("weight"), 2).alias("avg_weight"),
        F.round(F.stddev("weight"), 2).alias("std_weight"),
        F.round(F.sum("weight"), 2).alias("total_volume")
    )
    .orderBy("true_label")
)

print(f"\n  Total nodes   : {n_nodes:,}")
print(f"  Total edges   : {total_edges:,}")
print(f"  Graph density : {2 * total_edges / (n_nodes * (n_nodes - 1)):.6f}")
print(f"\n  Edge breakdown by true label:")
label_counts.show(truncate=False)

# Node degree distribution (top 10 highest-degree nodes)
degree_df = (
    edge_df.select(F.col("src").alias("node"))
    .union(edge_df.select(F.col("dst").alias("node")))
    .groupBy("node")
    .agg(F.count("*").alias("degree"))
    .orderBy(F.desc("degree"))
)

print("  Top 10 highest-degree nodes (likely fraud ring members):")
degree_df.show(10, truncate=False)

════════════════════════════════════════════════════════════
  GRAPH SUMMARY STATISTICS
════════════════════════════════════════════════════════════

  Total nodes   : 514
  Total edges   : 3,000
  Graph density : 0.022755

  Edge breakdown by true label:
+------------+----------+----------+----------+------------+
|true_label  |edge_count|avg_weight|std_weight|total_volume|
+------------+----------+----------+----------+------------+
|fraud_ring_A|28        |946.2     |17.07     |26493.56    |
|fraud_ring_B|15        |476.59    |6.92      |7148.79     |
|normal      |2957      |194.31    |389.08    |574586.53   |
+------------+----------+----------+----------+------------+

  Top 10 highest-degree nodes (likely fraud ring members):
+----+------+
|node|degree|
+----+------+
|136 |22    |
|195 |22    |
|486 |21    |
|361 |20    |
|370 |20    |
|38  |20    |
|284 |20    |
|233 |19    |
|59  |19    |
|170 |19    |
+----+------+
only showing top 10 rows



---

## ⚙️ Section 3: Power Iteration Clustering Model

### PIC Algorithm — Step by Step

Spark's `PowerIterationClustering` (Lin & Cohen, 2010) works as follows:

1. **Build the normalised affinity matrix** `A` from the edge list:
   $$A_{ij} = \frac{W_{ij}}{\sqrt{d_i \cdot d_j}} \quad \text{where } d_i = \sum_j W_{ij}$$
   This is the **symmetric normalised adjacency** (equivalent to `I - L_sym`).

2. **Initialise** the iteration vector `v⁰` uniformly or randomly.

3. **Iterate** until convergence (`‖v^(t+1) - v^(t)‖ < ε`):
   $$v^{(t+1)} \leftarrow \frac{A \cdot v^{(t)}}{\|A \cdot v^{(t)}\|_1}$$
   Each iteration is a **distributed sparse matrix-vector multiply** — trivially parallelised on Spark as a join + aggregation.

4. **Cluster the 1D scalar values** `v^(*)` using **k-means** with `k` clusters. Nodes with similar pseudo-eigenvector values belong to the same graph community.

### Hyperparameter Selection

| Parameter | Value | Rationale |
|---|---|---|
| `k` | `15` | Overestimate slightly; fraud rings form micro-clusters, normal traffic groups loosely. Use elbow/silhouette in production. |
| `maxIter` | `40` | PIC typically converges in 10–30 iterations. 40 provides buffer. |
| `weightCol` | `weight_norm` | Normalised weights [0,1] ensure stable float arithmetic across iterations. |
| `initMode` | `random` | `"degree"` init can bias toward high-degree hub nodes; random is safer for ring detection. |

> **Production tip:** In real deployments, determine optimal `k` via the **Eigengap heuristic** on the Laplacian spectrum, or use **Modularity maximisation** on a sample. PIC `k` is sensitive — fraud rings create gaps in the eigenspectrum that suggest the right `k`.

In [8]:
# ============================================================
# CELL 3.1 — Configure & fit PowerIterationClustering
# ============================================================

import time

print("🔄 Configuring PowerIterationClustering...")

# ── Model definition ─────────────────────────────────────────────────────────
pic = PowerIterationClustering(
    k=15,              # number of clusters (overestimate slightly)
    maxIter=40,        # max power iterations
    weightCol="weight_norm",  # normalised edge weights
    srcCol="src",      # must be LongType column
    dstCol="dst",      # must be LongType column
    initMode="random", # initialisation strategy
)

print(f"   k (clusters)    : {pic.getK()}")
print(f"   maxIter         : {pic.getMaxIter()}")
print(f"   weightCol       : {pic.getWeightCol()}")
print(f"   initMode        : {pic.getInitMode()}")

# ── Fit the model ─────────────────────────────────────────────────────────────
# Note: PIC.fit() returns a DataFrame of (id: LongType, cluster: IntegerType)
# There is no separate 'model' object — PIC is a transformer-style estimator.
print("\n⏳ Fitting PIC model (this triggers distributed power iterations)...")
t0 = time.time()

assignments_df = pic.assignClusters(edge_df)
assignments_df.cache()
n_assigned = assignments_df.count()   # materialise

elapsed = time.time() - t0

print(f"\n✅ PIC completed in {elapsed:.1f}s")
print(f"   Nodes assigned to clusters : {n_assigned:,}")
print(f"\n🔍 Sample cluster assignments (10 rows):")
assignments_df.orderBy("cluster", "id").show(10, truncate=False)

# ── Cluster size distribution ─────────────────────────────────────────────────
print("📊 Cluster size distribution:")
cluster_sizes = (
    assignments_df
    .groupBy("cluster")
    .agg(F.count("*").alias("node_count"))
    .orderBy("node_count")
)
cluster_sizes.show(15, truncate=False)

🔄 Configuring PowerIterationClustering...
   k (clusters)    : 15
   maxIter         : 40
   weightCol       : weight_norm
   initMode        : random

⏳ Fitting PIC model (this triggers distributed power iterations)...


26/04/10 12:50:47 WARN BlockManager: Block rdd_78_0 already exists on this machine; not re-adding it



✅ PIC completed in 13.4s
   Nodes assigned to clusters : 514

🔍 Sample cluster assignments (10 rows):
+---+-------+
|id |cluster|
+---+-------+
|1  |0      |
|2  |0      |
|3  |0      |
|4  |0      |
|5  |0      |
|6  |0      |
|8  |0      |
|9  |0      |
|10 |0      |
|11 |0      |
+---+-------+
only showing top 10 rows

📊 Cluster size distribution:
+-------+----------+
|cluster|node_count|
+-------+----------+
|12     |1         |
|9      |1         |
|4      |1         |
|8      |1         |
|10     |1         |
|5      |2         |
|2      |2         |
|6      |3         |
|11     |3         |
|1      |6         |
|13     |8         |
|7      |31        |
|14     |56        |
|0      |398       |
+-------+----------+



---

## 🚨 Section 4: Fraud Scoring & Ring Detection

### From Clusters to Fraud Scores

PIC gives us cluster membership, but not a fraud label. We now apply **cluster-level anomaly scoring** to separate fraud rings from normal transaction communities.

#### Fraud Ring Signature (Discriminating Features)

A fraud ring cluster exhibits a very specific combination of graph-theoretic properties:

| Metric | Normal Cluster | Fraud Ring Cluster |
|---|---|---|
| **Size** | Large (many nodes) | Micro (< 15 nodes) |
| **Internal edge density** | Low (sparse connections) | High → 1.0 (near-clique) |
| **Weight CV** (σ/μ) | High (varied spend) | Near 0 (scripted amounts) |
| **Volume-per-node ratio** | Low | Extremely high |
| **Degree regularity** | Irregular | All nodes same degree |

#### Composite Fraud Score

We compute a dimensionless **Fraud Ring Score** `S` for each cluster:

$$S = w_1 \cdot \rho_{\text{density}} + w_2 \cdot \frac{1}{1 + CV} + w_3 \cdot \log\left(1 + \frac{\text{volume}}{n_{\text{nodes}}}\right)_{\text{norm}}$$

Where:
- $\rho_{\text{density}}$ = internal edge density = `observed_edges / max_possible_edges` (max = `n*(n-1)/2`)
- $CV$ = coefficient of variation of edge weights within cluster = `σ/μ` (low CV → uniform scripted amounts)
- The volume-per-node term is log-normalised across all clusters

Clusters with `S > threshold` are **flagged as candidate fraud rings**.

In [9]:
# ============================================================
# CELL 4.1 — Join cluster assignments back to edges
# ============================================================

print("🔗 Joining cluster assignments to edge data...")

# Rename for join clarity
src_cluster = assignments_df.select(
    F.col("id").alias("src"),
    F.col("cluster").alias("src_cluster")
)
dst_cluster = assignments_df.select(
    F.col("id").alias("dst"),
    F.col("cluster").alias("dst_cluster")
)

# Join clusters to edges; keep only INTRA-cluster edges (same cluster on both ends)
# These are the edges that define each cluster's internal structure
enriched_edges = (
    edge_df
    .join(src_cluster, on="src", how="left")
    .join(dst_cluster, on="dst", how="left")
    .withColumn(
        "same_cluster",
        F.col("src_cluster") == F.col("dst_cluster")
    )
)

# Intra-cluster edges: used for density & weight-uniformity metrics
intra_cluster_edges = enriched_edges.filter(F.col("same_cluster") == True)
intra_cluster_edges.cache()

n_intra = intra_cluster_edges.count()
print(f"   Total edges    : {total_edges:,}")
print(f"   Intra-cluster  : {n_intra:,}  ({100*n_intra/total_edges:.1f}% of all edges)")
print(f"   Cross-cluster  : {total_edges - n_intra:,}")

print("\n🔍 Sample enriched edges:")
intra_cluster_edges.select(
    "src", "dst", "weight", "weight_norm", "true_label",
    "src_cluster", "dst_cluster", "same_cluster"
).show(8, truncate=False)

🔗 Joining cluster assignments to edge data...
   Total edges    : 3,000
   Intra-cluster  : 2,039  (68.0% of all edges)
   Cross-cluster  : 961

🔍 Sample enriched edges:
+---+---+------+-----------+----------+-----------+-----------+------------+
|src|dst|weight|weight_norm|true_label|src_cluster|dst_cluster|same_cluster|
+---+---+------+-----------+----------+-----------+-----------+------------+
|236|290|48.92 |0.004886   |normal    |0          |0          |true        |
|62 |483|203.47|0.020324   |normal    |0          |0          |true        |
|57 |217|378.38|0.037795   |normal    |0          |0          |true        |
|95 |323|776.08|0.077519   |normal    |0          |0          |true        |
|138|304|26.85 |0.002682   |normal    |0          |0          |true        |
|415|472|6.98  |6.97E-4    |normal    |0          |0          |true        |
|173|390|273.53|0.027322   |normal    |0          |0          |true        |
|330|469|241.36|0.024108   |normal    |0          |0        

In [10]:
# ============================================================
# CELL 4.2 — Compute cluster-level metrics
# ============================================================

print("📐 Computing cluster-level metrics...")

# ── Metric 1: Cluster node counts ────────────────────────────────────────────
node_counts = (
    assignments_df
    .groupBy("cluster")
    .agg(F.count("*").alias("n_nodes"))
)

# ── Metric 2: Intra-cluster edge stats ───────────────────────────────────────
edge_stats = (
    intra_cluster_edges
    .groupBy("src_cluster")
    .agg(
        F.count("*").alias("n_edges"),
        F.sum("weight").alias("total_volume"),
        F.avg("weight").alias("avg_weight"),
        F.stddev("weight").alias("std_weight"),
        F.min("weight").alias("min_weight"),
        F.max("weight").alias("max_weight"),
    )
    .withColumnRenamed("src_cluster", "cluster")
)

# ── Join metrics ─────────────────────────────────────────────────────────────
cluster_metrics = (
    node_counts
    .join(edge_stats, on="cluster", how="left")
    .fillna(0.0, subset=["n_edges", "total_volume", "avg_weight",
                          "std_weight", "min_weight", "max_weight"])
)

# ── Derived metrics ───────────────────────────────────────────────────────────
# Max possible edges in a cluster of n nodes: n*(n-1)/2
# Coefficient of Variation: std/mean (low → scripted/uniform amounts)
# Volume per node: total_volume / n_nodes

cluster_metrics = (
    cluster_metrics
    .withColumn(
        "max_possible_edges",
        (F.col("n_nodes") * (F.col("n_nodes") - 1) / 2).cast(DoubleType())
    )
    .withColumn(
        "internal_density",
        F.when(F.col("max_possible_edges") > 0,
               F.col("n_edges") / F.col("max_possible_edges"))
        .otherwise(0.0)
    )
    .withColumn(
        "coeff_variation",
        F.when(F.col("avg_weight") > 0,
               F.col("std_weight") / F.col("avg_weight"))
        .otherwise(999.0)
    )
    .withColumn(
        "volume_per_node",
        F.when(F.col("n_nodes") > 0,
               F.col("total_volume") / F.col("n_nodes"))
        .otherwise(0.0)
    )
)

# ── Normalise volume_per_node across clusters (for scoring) ──────────────────
vpn_max = cluster_metrics.agg(F.max("volume_per_node")).collect()[0][0]
cluster_metrics = cluster_metrics.withColumn(
    "volume_per_node_norm",
    F.when(vpn_max > 0, F.col("volume_per_node") / vpn_max).otherwise(0.0)
)

cluster_metrics.cache()

print("\n📊 Cluster metrics (ordered by internal density desc):")
(
    cluster_metrics
    .select(
        "cluster", "n_nodes", "n_edges", "max_possible_edges",
        F.round("internal_density", 4).alias("density"),
        F.round("coeff_variation", 4).alias("cv"),
        F.round("total_volume", 2).alias("total_vol"),
        F.round("volume_per_node", 2).alias("vol_per_node")
    )
    .orderBy(F.desc("density"), F.asc("n_nodes"))
    .show(15, truncate=False)
)

📐 Computing cluster-level metrics...


PySparkTypeError: [NOT_COLUMN] Argument `condition` should be a Column, got bool.

In [ ]:
# ============================================================
# CELL 4.3 — Composite Fraud Score & Flagging
# ============================================================

print("🎯 Computing composite Fraud Ring Score...")

# ── Weights for each signal component ────────────────────────────────────────
W_DENSITY   = 0.50   # density dominates — fraud rings are near-cliques
W_UNIFORMITY = 0.30  # low CV → scripted amounts
W_VOLUME    = 0.20   # high volume per node is secondary signal

# ── Low-CV score: maps CV→0 = score 1.0, CV→∞ = score 0.0 ───────────────────
# Formula: 1 / (1 + CV) — bounded in [0,1]
FRAUD_SCORE_THRESHOLD = 0.55   # tunable; flag clusters scoring above this
MICRO_CLUSTER_MAX_SIZE = 20    # hard filter: fraud rings are small

scored_clusters = (
    cluster_metrics
    .withColumn(
        "uniformity_score",
        F.lit(1.0) / (F.lit(1.0) + F.col("coeff_variation"))
    )
    .withColumn(
        "fraud_score",
        F.round(
            F.lit(W_DENSITY)    * F.col("internal_density")
          + F.lit(W_UNIFORMITY) * F.col("uniformity_score")
          + F.lit(W_VOLUME)     * F.col("volume_per_node_norm"),
        6)
    )
    .withColumn(
        "is_fraud_ring",
        (
            (F.col("fraud_score")  >= FRAUD_SCORE_THRESHOLD) &
            (F.col("n_nodes")      <= MICRO_CLUSTER_MAX_SIZE) &
            (F.col("internal_density") >= 0.80)   # must be nearly complete graph
        )
    )
    .orderBy(F.desc("fraud_score"))
)

scored_clusters.cache()

# ── Summary report ────────────────────────────────────────────────────────────
print("\n" + "═" * 80)
print("  FRAUD DETECTION RESULTS — SCORED CLUSTERS")
print("═" * 80)
(
    scored_clusters
    .select(
        "cluster", "n_nodes", "n_edges",
        F.round("internal_density", 4).alias("density"),
        F.round("coeff_variation", 4).alias("cv"),
        F.round("uniformity_score", 4).alias("uniformity"),
        F.round("volume_per_node", 2).alias("vol_per_node"),
        F.round("fraud_score", 4).alias("fraud_score"),
        "is_fraud_ring"
    )
    .show(15, truncate=False)
)

# ── Flagged fraud rings ───────────────────────────────────────────────────────
fraud_rings_found = scored_clusters.filter(F.col("is_fraud_ring") == True)
n_fraud_clusters  = fraud_rings_found.count()

print("\n" + "🚨 " * 20)
print(f"  DETECTED FRAUD RINGS: {n_fraud_clusters}")
print("🚨 " * 20)
fraud_rings_found.select(
    "cluster", "n_nodes", "n_edges",
    F.round("internal_density", 4).alias("density"),
    F.round("coeff_variation", 4).alias("cv"),
    F.round("fraud_score", 4).alias("fraud_score"),
    F.round("total_volume", 2).alias("total_volume"),
    F.round("avg_weight", 2).alias("avg_txn_amount")
).show(truncate=False)

In [ ]:
# ============================================================
# CELL 4.4 — Node-level fraud flag & ground truth evaluation
# ============================================================

print("🔎 Mapping fraud cluster flags back to individual nodes...")

# Extract flagged cluster IDs
flagged_cluster_ids = [
    row["cluster"] for row in fraud_rings_found.select("cluster").collect()
]
print(f"   Flagged cluster IDs: {flagged_cluster_ids}")

# Build node-level result: join assignment + fraud flag
flagged_clusters_df = scored_clusters.select(
    "cluster",
    F.round("fraud_score", 4).alias("fraud_score"),
    "is_fraud_ring"
)

node_results = (
    assignments_df
    .join(flagged_clusters_df, on="cluster", how="left")
    .withColumnRenamed("id", "node_id")
    .withColumn("is_fraud_ring", F.coalesce(F.col("is_fraud_ring"), F.lit(False)))
)

# ── Attach ground truth label for evaluation ──────────────────────────────────
# Build node→true_label lookup from the edge list
src_labels = edge_df.select(
    F.col("src").alias("node_id"), F.col("true_label")
)
dst_labels = edge_df.select(
    F.col("dst").alias("node_id"), F.col("true_label")
)
node_labels = (
    src_labels.union(dst_labels)
    .filter(F.col("true_label") != "normal")
    .distinct()
)

node_results_gt = (
    node_results
    .join(node_labels, on="node_id", how="left")
    .withColumn(
        "true_fraud",
        F.col("true_label").isNotNull() & (F.col("true_label") != "normal")
    )
)

node_results_gt.cache()

# ── Confusion matrix counts ───────────────────────────────────────────────────
tp = node_results_gt.filter(F.col("is_fraud_ring") & F.col("true_fraud")).count()
fp = node_results_gt.filter(F.col("is_fraud_ring") & ~F.col("true_fraud")).count()
fn = node_results_gt.filter(~F.col("is_fraud_ring") & F.col("true_fraud")).count()
tn = node_results_gt.filter(~F.col("is_fraud_ring") & ~F.col("true_fraud")).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "═" * 60)
print("  MODEL EVALUATION — GROUND TRUTH vs PREDICTIONS")
print("═" * 60)
print(f"\n  Confusion Matrix:")
print(f"  ┌─────────────────────────┬────────────┬────────────┐")
print(f"  │                         │ Pred: FRAUD│ Pred: LEGIT│")
print(f"  ├─────────────────────────┼────────────┼────────────┤")
print(f"  │ True: FRAUD             │ TP = {tp:<6} │ FN = {fn:<6} │")
print(f"  │ True: LEGIT             │ FP = {fp:<6} │ TN = {tn:<6} │")
print(f"  └─────────────────────────┴────────────┴────────────┘")
print(f"\n  Precision : {precision:.4f}  ({precision*100:.1f}%)")
print(f"  Recall    : {recall:.4f}  ({recall*100:.1f}%)")
print(f"  F1 Score  : {f1:.4f}")

print("\n🔍 Flagged fraud nodes (sample):")
(
    node_results_gt
    .filter(F.col("is_fraud_ring") == True)
    .select("node_id", "cluster", "fraud_score", "true_label", "true_fraud")
    .orderBy("cluster", "node_id")
    .show(20, truncate=False)
)

In [ ]:
# ============================================================
# CELL 4.5 — Full pipeline summary report
# ============================================================

print("\n" + "═" * 70)
print("  PIPELINE EXECUTION SUMMARY")
print("═" * 70)

total_nodes_flagged = node_results_gt.filter(F.col("is_fraud_ring")).count()
total_vol_flagged   = (
    intra_cluster_edges
    .join(
        fraud_rings_found.select("cluster"),
        intra_cluster_edges.src_cluster == fraud_rings_found.cluster,
        how="inner"
    )
    .agg(F.sum("weight"))
    .collect()[0][0] or 0.0
)

print(f"""
  Graph Statistics:
    - Total nodes                   : {n_nodes:,}
    - Total edges                   : {total_edges:,}
    - PIC clusters (k)              : {pic.getK()}

  Detection Results:
    - Fraud ring clusters flagged   : {n_fraud_clusters}
    - Fraud cluster IDs             : {flagged_cluster_ids}
    - Fraud nodes flagged           : {total_nodes_flagged}
    - Suspicious transaction volume : ${total_vol_flagged:,.2f}

  Model Quality:
    - True Positives (TP)           : {tp}
    - False Positives (FP)          : {fp}
    - False Negatives (FN)          : {fn}
    - Precision                     : {precision:.4f}
    - Recall                        : {recall:.4f}
    - F1 Score                      : {f1:.4f}

  Configuration:
    - Fraud Score Threshold         : {FRAUD_SCORE_THRESHOLD}
    - Max Micro-Cluster Size        : {MICRO_CLUSTER_MAX_SIZE}
    - Density Weight (w1)           : {W_DENSITY}
    - Uniformity Weight (w2)        : {W_UNIFORMITY}
    - Volume Weight (w3)            : {W_VOLUME}
""")

print("═" * 70)

---

## 🏭 Section 5: Production Considerations & Next Steps

### Scaling to Production

This notebook demonstrates the **core algorithmic pipeline**. In a production environment, several additional engineering decisions apply:

#### 1. Optimal `k` Selection

In production, `k` should not be hard-coded. Use the **Eigengap Heuristic**:
- Compute the first `k_max` eigenvalues of `L_sym` on a graph **sample** (e.g., 1% via stratified sampling)
- The optimal `k` is at the largest gap: `k* = argmax_i (λ_{i+1} - λ_i)`
- Alternatively: run PIC with `k ∈ {5, 10, 15, 20, 25}` and select via **Silhouette Score** on the 1D pseudo-eigenvector values

#### 2. Dynamic Graph Updates (Streaming)

Fraud detection must respond to new transactions in near real-time:
- Use **Spark Structured Streaming** to ingest from Kafka
- Maintain a **sliding window** (e.g., 7-day) graph
- Re-run PIC periodically (e.g., hourly micro-batch) on the window
- Incrementally score new nodes by projecting onto the existing pseudo-eigenvector (avoid full refit)

#### 3. Feature Engineering Enhancements

Enrich the edge weight `W` with additional signals before PIC:

```
W_composite = α·(amount_norm) + β·(time_proximity) + γ·(geo_closeness) + δ·(velocity_flag)
```

Higher composite weights pull fraudulent nodes closer in pseudo-eigenvector space.

#### 4. Alerting Pipeline

```
PIC Output → Fraud Scorer → Delta Lake (fraud_alerts table)
                         → Kafka Topic (real-time case creation)
                         → Case Management System (JIRA / ServiceNow)
```

#### 5. Model Monitoring

Track drift in:
- `internal_density` of flagged clusters over time (should remain high for true rings)
- FP rate from analyst feedback loop
- Graph diameter and clustering coefficient trends

---

### Summary: Why PIC Over Alternatives?

| Algorithm | Complexity | Distributed? | Fraud Ring Suitability |
|---|---|---|---|
| Naive k-means on features | `O(n·k·t)` | ✅ (Spark MLlib) | ❌ No graph structure |
| Classical Spectral (dense) | `O(n³)` | ❌ | ✅ Optimal but unscalable |
| Louvain Modularity | `O(n log n)` | ⚠️ Partial | ✅ Good for communities |
| **PIC (this notebook)** | **`O(|E|·t)`** | **✅ Native Spark** | **✅ Optimal** |
| GraphX Pregel | `O(|E|·t)` | ✅ | ✅ More flexible, more code |

PIC hits the **exact sweet spot** for fraud ring detection: it uses the graph Laplacian's spectral properties (rigorous mathematics) while remaining fully distributed and `O(edges)` per iteration (production-scale engineering).

In [ ]:
# ============================================================
# CELL 5.1 — Graceful teardown
# ============================================================

print("🧹 Releasing cached DataFrames...")

for df in [edge_df, intra_cluster_edges, assignments_df,
           cluster_metrics, scored_clusters, node_results_gt]:
    try:
        df.unpersist()
    except Exception:
        pass

print("✅ Cache cleared.")
print("\n🏁 Pipeline complete. SparkSession remains active for further exploration.")
print(f"   Spark UI: {spark.sparkContext.uiWebUrl}")